In [ ]:
!git clone https://github.com/GyanRout/fraud_detection_pipeline.git

In [ ]:
!pip install -r 

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import time
import os

# ==========================================
# 1. Synthetic Data Generation
# ==========================================
def generate_fraud_data(n_samples=50000, fraud_ratio=0.02):
    """
    Generates synthetic transaction data.
    Features: 
    - amount: Transaction value.
    - velocity_1h: Number of transactions in the last hour.
    - time_of_day_sin / time_of_day_cos: Cyclical time encoding.
    - device_risk_score: Aggregated score from browser fingerprints.
    """
    np.random.seed(42)
    
    n_fraud = int(n_samples * fraud_ratio)
    n_legit = n_samples - n_fraud
    
    # Legit transactions (Normal distributions)
    legit_amount = np.random.exponential(scale=50, size=n_legit)
    legit_velocity = np.random.poisson(lam=1, size=n_legit)
    legit_risk = np.random.normal(loc=0.2, scale=0.1, size=n_legit)
    
    # Fraud transactions (Anomalous distributions)
    fraud_amount = np.random.exponential(scale=500, size=n_fraud)
    fraud_velocity = np.random.poisson(lam=10, size=n_fraud)
    fraud_risk = np.random.normal(loc=0.8, scale=0.1, size=n_fraud)
    
    # Combine and create labels
    amounts = np.concatenate([legit_amount, fraud_amount])
    velocities = np.concatenate([legit_velocity, fraud_velocity])
    risks = np.concatenate([legit_risk, fraud_risk])
    
    # Time encoding (cyclical continuous variables)
    hours = np.random.uniform(0, 24, n_samples)
    time_sin = np.sin(2 * np.pi * hours / 24)
    time_cos = np.cos(2 * np.pi * hours / 24)
    
    y = np.concatenate([np.zeros(n_legit), np.ones(n_fraud)])
    
    X = np.column_stack((amounts, velocities, time_sin, time_cos, risks))
    return X, y

# ==========================================
# 2. PyTorch Dataset & Model Definition
# ==========================================
class TransactionDataset(Dataset):
    def __init__(self, features, labels):
        self.features = torch.FloatTensor(features)
        self.labels = torch.FloatTensor(labels).unsqueeze(1)
        
    def __len__(self):
        return len(self.features)
        
    def __getitem__(self, idx):
        return self.features[idx], self.labels[idx]

class FraudDetectionMLP(nn.Module):
    """
    Highly optimized Multilayer Perceptron for sub-millisecond inference.
    Uses ELU activations to avoid dead neurons common in imbalanced tabular data.
    """
    def __init__(self, input_dim):
        super(FraudDetectionMLP, self).__init__()
        # We use a shallow, wide network to minimize sequential layer computation time
        self.network = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.BatchNorm1d(64),
            nn.ELU(),
            nn.Dropout(0.2),
            
            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.ELU(),
            nn.Dropout(0.2),
            
            nn.Linear(32, 1),
            nn.Sigmoid() # Outputs a probability [0, 1]
        )
        
    def forward(self, x):
        return self.network(x)

# ==========================================
# 3. Focal Loss Implementation
# ==========================================
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.bce = nn.BCELoss(reduction='none')
        
    def forward(self, inputs, targets):
        bce_loss = self.bce(inputs, targets)
        pt = torch.exp(-bce_loss) # derived from BCE definition
        focal_loss = self.alpha * (1 - pt) ** self.gamma * bce_loss
        return focal_loss.mean()

# ==========================================
# 4. Training Loop & TorchScript Export
# ==========================================
def train_and_export_model():
    # 1. Data Prep
    print("Generating data...")
    X, y = generate_fraud_data()
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)
    
    # Crucial: Save the scaler logic for the API later. 
    # In production, we'd save this scaler object. Here, we fit on train.
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    
    train_loader = DataLoader(TransactionDataset(X_train_scaled, y_train), batch_size=256, shuffle=True)
    
    # 2. Setup Device (Colab provides CUDA)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Training on device: {device}")
    
    model = FraudDetectionMLP(input_dim=5).to(device)
    criterion = FocalLoss(alpha=0.75, gamma=2.0) # Higher alpha focuses on minority class
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    
    # 3. Training Loop
    epochs = 20
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            
        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs} | Loss: {epoch_loss/len(train_loader):.4f}")
            
    # 4. Compile to TorchScript for Production
    print("Compiling model via TorchScript...")
    model.eval()
    model.to("cpu") # Force back to CPU for API deployment
    
    # Create a dummy input tensor matching our feature dimension
    example_input = torch.randn(1, 5)
    
    # Trace the model (converts Python code to static C++ graph)
    traced_model = torch.jit.trace(model, example_input)
    
    # Save the artifact
    os.makedirs("models", exist_ok=True)
    save_path = "content/fraud_detection_pipeline/smodels/fraud_model_traced.pt"
    traced_model.save(save_path)
    print(f"Enterprise-ready model saved to: {save_path}")

train_and_export_model()